In [2]:
# Sekcje 1: Importowanie bibliotek

import pandas as pd # tabele danych
import seaborn as sns # źródło danych
import matplotlib.pyplot as plt # wykresy
from scipy import stats # statystyki

from sklearn.model_selection import train_test_split # podzielenie danych na treningowe i testowe
from sklearn.preprocessing import StandardScaler # wyrównywacz skali, że model miał łatwiej się uczyć
from sklearn.linear_model import LogisticRegression # model (regresja logistyczna)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # metryki:
# accuracy (dokładność),
# classification_report (precision, recall, f1-score, support [ile było przykładów]),
# confusion_matrix (macierz pomyłek)

import numpy as np # obliczenia

In [3]:
# Sekcja 2: Wczytanie danych i wstępne przygotowanie

titanic = sns.load_dataset("titanic")

print("Pierwsze 10 wierszy: ")
print(titanic.head(10))

Pierwsze 10 wierszy: 
   survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
0         0       3    male  22.0      1      0   7.2500        S   Third   
1         1       1  female  38.0      1      0  71.2833        C   First   
2         1       3  female  26.0      0      0   7.9250        S   Third   
3         1       1  female  35.0      1      0  53.1000        S   First   
4         0       3    male  35.0      0      0   8.0500        S   Third   
5         0       3    male   NaN      0      0   8.4583        Q   Third   
6         0       1    male  54.0      0      0  51.8625        S   First   
7         0       3    male   2.0      3      1  21.0750        S   Third   
8         1       3  female  27.0      0      2  11.1333        S   Third   
9         1       2  female  14.0      1      0  30.0708        C  Second   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False  

| Kolumna | Znaczenie |
|---|---|
| `survived` | Czy pasażer przeżył: `0` = nie, `1` = tak |
| `pclass` | Klasa biletu: `1` = pierwsza, `2` = druga, `3` = trzecia |
| `sex` | Płeć pasażera: `male` = mężczyzna, `female` = kobieta |
| `age` | Wiek pasażera |
| `sibsp` | Liczba rodzeństwa lub małżonków na statku |
| `parch` | Liczba rodziców lub dzieci na statku |
| `fare` | Cena biletu |
| `embarked` | Port wejścia na statek zapisany skrótem (`S` - Southampton, `C` - Cherbourg, `Q` - Queenstown)|
| `class` | Klasa biletu zapisana słownie: `First`, `Second`, `Third` |
| `who` | Typ osoby: `man`, `woman` lub `child` |
| `adult_male` | Czy pasażer był dorosłym mężczyzną: `True` / `False` |
| `deck` | Pokład/statkowa sekcja, np. `A`, `B`, `C`; często brakuje tej wartości |
| `embark_town` | Miasto/port wejścia na statek |
| `alive` | Czy pasażer przeżył zapisane tekstowo: `yes` / `no` |
| `alone` | Czy pasażer podróżował sam: `True` / `False` |

In [9]:
print("\nBrakujące wartości w każdej kolumnie:")
print(titanic.isnull().sum())

print("="* 50)

# sprawdzenei liczby wierszy przed czyszczeniem
liczba_wierszy_przed = len(titanic)

# czyszzcenie (usunięcie wierszy nz age=NaN)
titanic_clean = titanic.dropna(subset=["age"])

# ile wierszy po czyszceniu
liczba_wierszy_po = len(titanic_clean)

print("\nLiczba wierszy przed czyszczeniem:", liczba_wierszy_przed)
print("Liczba wierszy po czyszczeniu:", liczba_wierszy_po)
print("Usunięto wierszy:", liczba_wierszy_przed - liczba_wierszy_po)


print("="* 50)
print(titanic_clean.isnull().sum())


Brakujące wartości w każdej kolumnie:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

Liczba wierszy przed czyszczeniem: 891
Liczba wierszy po czyszczeniu: 714
Usunięto wierszy: 177
survived         0
pclass           0
sex              0
age              0
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           530
embark_town      2
alive            0
alone            0
dtype: int64


In [10]:
# Sekcja 3: Kategoryzacja biletów (modyfikacja)

# - `0` — niska cena,
# - `1` — średnia cena,
# - `2` — wysoka cena.

def categorize_fare(fare, fare_ranges):
  if fare <= fare_ranges[0]:
    return 0
  elif fare <= fare_ranges[1]:
    return 1
  else:
    return 2

min_fare = titanic_clean["fare"].min()
max_fare = titanic_clean["fare"].max()

step = (max_fare - min_fare) / 3

fare_ranges = [
    min_fare + step,
    min_fare + 2 * step
]


# Wyświetlenie przedziałów cenowych
print("\nPrzedziały cenowe biletów:")
print(f"Niska: {min_fare:.2f} - {fare_ranges[0]:.2f}")
print(f"Średnia: {fare_ranges[0]:.2f} - {fare_ranges[1]:.2f}")
print(f"Wysoka: {fare_ranges[1]:.2f} - {max_fare:.2f}")


Przedziały cenowe biletów:
Niska: 0.00 - 170.78
Średnia: 170.78 - 341.55
Wysoka: 341.55 - 512.33


In [11]:
# Sekcja 4: Dokakładne przygotwnaie danych

# Bo nie każda kolumna pomaga modelowi, bo niektóre dane są:
# - przydatne, bo mają związek z przeżyciem,
# - zbędne, bo powtarzają informacje z innych kolumn,
# - problematyczne, bo mają dużo braków,
# - zakazane, bo zawierają odpowiedź.

# [najważnijesze cechy: sex, pcalss, age, fare_category (fare zmienione na nasze kategorie)]


titanic_model = titanic_clean.copy()

titanic_model["sex"] = titanic_model["sex"].map({
    "male" : 0,
    "female" : 1
})

titanic_model["fare_category"] = titanic_model["fare"].apply(
    lambda fare: categorize_fare(fare, fare_ranges)
)

print(titanic_model["fare_category"].value_counts())

# Wybór cech, które będą używane przez model
selected_features = ["sex", "pclass", "age", "fare_category"]

# Podgląd przygotowanych danych
print(titanic_model[selected_features].head(10))

fare_category
0    696
1     15
2      3
Name: count, dtype: int64
    sex  pclass   age  fare_category
0     0       3  22.0              0
1     1       1  38.0              0
2     1       3  26.0              0
3     1       1  35.0              0
4     0       3  35.0              0
6     0       1  54.0              0
7     0       3   2.0              0
8     1       3  27.0              0
9     1       2  14.0              0
10    1       3   4.0              0
